<font color='red'><b>**WARNING**</b></font> <br/>
어떠한 사유로도 임의로 복사, 촬영, 녹음, 복제, 보관, 전송하거나 허가 받지 않은 저장매체를 이용한 보관, 제3자에게 누설, 공개 또는 사용하는 등의 무단 사용 및 불법 배포 시 법적 조치를 받을 수 있습니다. <br/>

<div style="text-align: right; color: #7f8c8d; font-size: 0.9em; margin-top: 20px;">
📝 Author: 박사홍 (Sahong Pak)</br>
📧 Contact: sahong.pak@gmail.com</br>
📌 Version: v2.0</br>
📅 Last Updated: 2026-03-12</br>
</div>

</br>

# 학습 내용
>이번 장에서는 <strong>아키텍처별 모델(Model Architectures)</strong>에 대해 학습합니다.</br></br>
>Encoder-only, Decoder-only, Encoder-Decoder 구조의 차이와 내부 구성요소를 학습해봅시다.

</br>

# Transformer 아키텍처 분류
> 현대 NLP 모델은 Transformer의 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">어떤 부분을 사용하느냐</mark>에 따라 세 가지로 분류됩니다.

어텐션 방향이 태스크의 본질을 결정합니다. Transformer Self-Attention은 시퀀스 내 모든 토큰 쌍의 관계를 병렬로 계산하는 핵심 연산인데, 양방향으로 앞뒤 문맥을 모두 참조할지, 단방향(Causal)으로 이전 토큰만 참조할지에 따라 적합한 태스크가 달라집니다.</br>

Encoder-only(BERT 계열)는 감성 분류, 개체명 인식(NER), 질의응답처럼 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">주어진 텍스트를 분석</mark>하는 이해 태스크에 최적입니다. "은행에 갔다"에서 "은행"이 금융기관인지 강가인지 판단하려면 문장 전체의 양방향 문맥이 필요하기 때문입니다. Decoder-only(GPT 계열)는 텍스트 생성에 최적인데, 생성은 본질적으로 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">왼쪽에서 오른쪽으로 이미 생성된 토큰만 참조</mark>하며 다음 토큰을 예측하는 과정이므로, 마스킹(Causal Mask)으로 미래 토큰을 가리는 단방향 구조가 정확히 일치합니다. 단순한 구조 덕분에 모델을 극단적으로 크게 키울 수 있어 스케일링에도 유리합니다. Encoder-Decoder(T5, BART 계열)는 번역이나 요약처럼 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">입력 전체를 이해한 뒤 새로운 시퀀스를 생성</mark>해야 하는 변환 태스크에 적합하며, 인코더가 양방향으로 입력을 완전히 이해하고 디코더가 그 표현을 바탕으로 출력을 생성합니다. 태스크의 성격(이해/생성/변환)이 아키텍처 선택의 기준입니다.

<table style="width:100%">
  <thead>
    <tr>
      <th style="text-align:center">아키텍처</th>
      <th style="text-align:center">대표 모델</th>
      <th>용도</th>
      <th style="text-align:center">어텐션 방향</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:center">Encoder-only</td><td style="text-align:center"><mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">BERT</mark></td><td>분류, NER, QA</td><td style="text-align:center">양방향</td></tr>
    <tr><td style="text-align:center">Decoder-only</td><td style="text-align:center"><mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">GPT</mark></td><td>텍스트 생성</td><td style="text-align:center">단방향 (좌→우)</td></tr>
    <tr><td style="text-align:center">Encoder-Decoder</td><td style="text-align:center"><mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">T5</mark>, BART</td><td>번역, 요약</td><td style="text-align:center">양방향 + 단방향</td></tr>
  </tbody>
</table>

</br>

## Encoder-only (BERT 계열)
> 입력 전체를 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">양방향으로 동시에</mark> 처리하여 문맥을 이해합니다.

In [ ]:
# TODO 1: 사전학습된 BERT 토크나이저와 모델을 불러와 "Hello, world!"를 입력하고, 마지막 은닉 상태의 shape를 출력해봅시다.

from transformers import AutoModel, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModel.from_pretrained("bert-base-uncased")

inputs = tokenizer("Hello, world!", return_tensors="pt")
outputs = model(**inputs)
print(f"last_hidden_state: {outputs.last_hidden_state.shape}")

💡BERT의 학습 방식
> Masked Language Modeling(MLM): 입력 토큰의 15%를 마스킹하고 예측</br>
> Next Sentence Prediction(NSP): 두 문장의 연결 관계 예측

</br>

## Decoder-only (GPT 계열)
> <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">이전 토큰만 참조</mark>하며 다음 토큰을 순차적으로 생성합니다.

In [ ]:
# TODO 2: 사전학습된 GPT-2 토크나이저와 인과 언어 모델을 불러와 "The future of AI is"를 입력하고 텍스트를 생성해봅시다.

from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("gpt2")
model = AutoModelForCausalLM.from_pretrained("gpt2")

inputs = tokenizer("The future of AI is", return_tensors="pt")
outputs = model.generate(inputs["input_ids"], max_length=20)
print(tokenizer.decode(outputs[0]))

</br>

# Transformer 핵심 구성요소

## Layer Normalization
> 각 레이어의 출력을 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">정규화하여 학습을 안정화</mark>합니다.

In [ ]:
# TODO 3: 레이어 정규화를 생성하고, 랜덤 입력에 대해 정규화 후 평균과 표준편차를 출력해봅시다.

import torch
import torch.nn as nn

hidden_dim = 768
layer_norm = nn.LayerNorm(hidden_dim)

x = torch.randn(1, 5, hidden_dim)   # (batch, seq, hidden)
output = layer_norm(x)
print(f"입력 평균: {x[0, 0, :5].data.round(decimals=3)}")
print(f"출력 평균: {output.mean(dim=-1).data.round(decimals=4)}")
print(f"출력 표준편차: {output.std(dim=-1).data.round(decimals=4)}")

💡LayerNorm vs BatchNorm
> BatchNorm: 배치 차원에서 정규화 (배치 크기에 의존)</br>
> LayerNorm: <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">피처 차원에서 정규화</mark> (배치 크기 독립, 시퀀스에 적합)

</br>

## Residual Connection (잔차 연결)
> 입력을 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">출력에 더하여</mark> 기울기 소실을 방지하고 깊은 네트워크 학습을 가능하게 합니다.

In [ ]:
# TODO 4: Pre-LN 패턴으로, 정규화 후 어텐션을 적용하여 잔차 연결하고, 정규화 후 MLP를 적용하여 잔차 연결하는 코드를 작성해봅시다.

x = x + self.attention(self.norm1(x))
x = x + self.mlp(self.norm2(x))

</br>

## Position Embedding (위치 임베딩)
> Transformer는 순서 정보가 없으므로 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">토큰의 위치 정보를 별도로 주입</mark>합니다.

<table style="width:100%">
  <thead>
    <tr>
      <th style="text-align:center">방식</th>
      <th>설명</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:center">고정 (Sinusoidal)</td><td>삼각함수로 위치 계산 (학습 불필요)</td></tr>
    <tr><td style="text-align:center"><mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">학습 (Learned)</mark></td><td>위치별 임베딩을 학습 (BERT, GPT)</td></tr>
    <tr><td style="text-align:center">상대적 (Relative)</td><td>토큰 간 거리 기반 (T5)</td></tr>
  </tbody>
</table>

In [ ]:
# TODO 5: 토큰 임베딩 클래스를 정의하여 토큰 임베딩과 위치 임베딩을 초기화하고, 순전파에서 두 임베딩을 더하여 반환한 뒤 테스트해봅시다.

class TokenEmbedding(nn.Module):
    def __init__(self, vocab_size, hidden_dim, max_len=512):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, hidden_dim)
        self.pos_emb = nn.Embedding(max_len, hidden_dim)

    def forward(self, x):
        positions = torch.arange(x.size(1), device=x.device)
        return self.token_emb(x) + self.pos_emb(positions)

embed = TokenEmbedding(vocab_size=30000, hidden_dim=768)
x = torch.LongTensor([[1, 42, 256]])
output = embed(x)
print(f"입력: {x.shape}")
print(f"출력 (토큰+위치): {output.shape}")

</br>

## Feed-Forward Network (MLP)
> 어텐션 출력을 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">비선형 변환</mark>하는 2층 완전 연결 네트워크입니다.

In [ ]:
# TODO 6: 피드포워드 네트워크 클래스를 정의하여 두 개의 선형 레이어와 활성화 함수를 초기화하고, 순전파를 구현한 뒤 테스트해봅시다.

class FeedForward(nn.Module):
    def __init__(self, hidden_dim, ff_dim):
        super().__init__()
        self.fc1 = nn.Linear(hidden_dim, ff_dim)
        self.fc2 = nn.Linear(ff_dim, hidden_dim)
        self.gelu = nn.GELU()

    def forward(self, x):
        return self.fc2(self.gelu(self.fc1(x)))

ff = FeedForward(hidden_dim=768, ff_dim=3072)
x = torch.randn(1, 5, 768)
output = ff(x)
print(f"입력: {x.shape} → 출력: {output.shape}")
print(f"파라미터 수: {sum(p.numel() for p in ff.parameters()):,}")

💡왜 Decoder-only 모델이 대세인가?
> 스케일링 법칙에서 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">모델 크기에 비례하여 성능이 향상</mark>되는 것이 확인되었습니다.</br>
> 단순한 구조로 대규모 학습이 용이하며, 프롬프트만으로 다양한 태스크를 수행합니다.

</br>

# 임베딩 시각화
> 임베딩 벡터를 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">PCA로 2D 평면에 투영</mark>하여 단어 간 관계를 시각적으로 확인합니다.

워드 임베딩(Word Embedding)은 단어를 고정 차원의 밀집 벡터(dense vector)로 변환하는 과정입니다. 잘 학습된 임베딩은 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">의미가 유사한 단어끼리 가까운 위치</mark>에 배치됩니다.

고차원 임베딩 벡터를 사람이 직접 해석하기 어려우므로, <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">PCA(주성분 분석)</mark>로 2차원으로 축소하여 scatter plot으로 시각화합니다.

<table style="width:100%">
  <thead>
    <tr>
      <th style="text-align:center">그룹</th>
      <th style="text-align:center">단어</th>
      <th style="text-align:center">색상</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:center">동물</td><td style="text-align:center">cat, dog, fish</td><td style="text-align:center">빨강</td></tr>
    <tr><td style="text-align:center">과일</td><td style="text-align:center">apple, banana, orange</td><td style="text-align:center">파랑</td></tr>
    <tr><td style="text-align:center">국가</td><td style="text-align:center">korea, japan, france</td><td style="text-align:center">초록</td></tr>
  </tbody>
</table>

In [ ]:
# TODO 7: 단어 목록을 정의하고 nn.Embedding으로 임베딩 벡터를 생성해봅시다.

import torch
import torch.nn as nn

words = ["cat", "dog", "fish", "apple", "banana", "orange", "korea", "japan", "france"]
word_to_id = {word: idx for idx, word in enumerate(words)}

vocab_size = len(words)
embedding_dim = 64

embedding_layer = nn.Embedding(vocab_size, embedding_dim)

input_ids = torch.tensor([word_to_id[w] for w in words])
embedding_vectors = embedding_layer(input_ids)

print(f"단어 수: {vocab_size}")
print(f"임베딩 차원: {embedding_dim}")
print(f"임베딩 벡터 shape: {embedding_vectors.shape}")

In [ ]:
# TODO 8: PCA로 임베딩 벡터를 2차원으로 축소하고 scatter plot으로 시각화해봅시다.

import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

embedding_2d = PCA(n_components=2).fit_transform(embedding_vectors.detach().numpy())

group_colors = {
    "동물": (["cat", "dog", "fish"], "red"),
    "과일": (["apple", "banana", "orange"], "blue"),
    "국가": (["korea", "japan", "france"], "green"),
}

plt.figure(figsize=(8, 6))
for group_name, (group_words, color) in group_colors.items():
    indices = [words.index(w) for w in group_words]
    plt.scatter(embedding_2d[indices, 0], embedding_2d[indices, 1],
                c=color, label=group_name, s=100)
    for idx in indices:
        plt.annotate(words[idx], (embedding_2d[idx, 0], embedding_2d[idx, 1]),
                     fontsize=11, ha="center", va="bottom", xytext=(0, 8),
                     textcoords="offset points")

plt.title("nn.Embedding 벡터의 PCA 2D 시각화 (학습 전)")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

💡학습 전 vs 학습 후 임베딩
> 위 시각화에서 단어들이 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">그룹별로 뭉치지 않는 것은 정상</mark>입니다. `nn.Embedding`은 랜덤 초기화된 상태이므로, 학습 전에는 의미 관계가 반영되지 않습니다.</br>
> 사전학습된 모델(BERT, GPT 등)의 임베딩은 대규모 말뭉치에서 학습되어 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">유사한 단어끼리 가까이 위치</mark>합니다.

</br>

# 코사인 유사도 (Cosine Similarity)
> 두 벡터 간의 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">방향의 유사성</mark>을 측정하는 지표입니다. 값이 1에 가까울수록 유사하고, 0에 가까울수록 무관합니다.

코사인 유사도는 벡터의 크기(norm)가 아닌 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">각도</mark>만을 기준으로 유사성을 판단합니다. 수식은 다음과 같습니다.

$$\text{cosine\_similarity}(\mathbf{a}, \mathbf{b}) = \frac{\mathbf{a} \cdot \mathbf{b}}{\|\mathbf{a}\| \times \|\mathbf{b}\|}$$

> 임베딩 공간에서 가까운 거리에 있다고 반드시 유사한 것은 아닙니다. 벡터 공간에서는 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">거리보다 방향과 각도가 더 중요</mark>합니다.

In [ ]:
# TODO 9: F.cosine_similarity로 단어 쌍의 코사인 유사도를 계산해봅시다.

import torch.nn.functional as F

pairs = [("cat", "dog"), ("cat", "apple"), ("korea", "japan"), ("banana", "france")]

for word_a, word_b in pairs:
    vec_a = embedding_vectors[word_to_id[word_a]]
    vec_b = embedding_vectors[word_to_id[word_b]]
    similarity = F.cosine_similarity(vec_a.unsqueeze(0), vec_b.unsqueeze(0))
    print(f"cosine_similarity({word_a}, {word_b}) = {similarity.item():.4f}")

In [ ]:
# TODO 10: 전체 단어 간 코사인 유사도 히트맵을 시각화해봅시다.

import matplotlib.pyplot as plt

similarity_matrix = F.cosine_similarity(
    embedding_vectors.unsqueeze(0),
    embedding_vectors.unsqueeze(1),
    dim=-1
).detach().numpy()

plt.figure(figsize=(8, 6))
plt.imshow(similarity_matrix, cmap="RdYlBu_r", vmin=-1, vmax=1)
plt.colorbar(label="Cosine Similarity")
plt.xticks(range(len(words)), words, rotation=45, ha="right")
plt.yticks(range(len(words)), words)
plt.title("단어 간 코사인 유사도 히트맵 (학습 전)")
plt.tight_layout()
plt.show()

💡코사인 유사도 해석
> 학습 전이므로 같은 그룹 내 단어끼리 유사도가 높지 않을 수 있습니다. 이는 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">임베딩이 랜덤 초기화</mark>되었기 때문입니다.</br>
> 학습이 진행되면 "cat"과 "dog"처럼 의미가 가까운 단어의 유사도는 올라가고, "cat"과 "france"처럼 무관한 단어의 유사도는 낮아집니다.</br>
> `F.cosine_similarity(a, b, dim)`에서 `dim`은 유사도를 계산할 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">벡터 차원 축</mark>을 지정합니다.